In [1]:
import pandas as pd

In [2]:
df_alcohol = pd.read_csv("health/alcohol.csv")

In [3]:
df_alcohol.head()

,IndicatorCode,Indicator,ValueType,ParentLocationCode,ParentLocation,Location type,SpatialDimValueCode,Location,Period type,Period,...,FactValueUoM,FactValueNumericLowPrefix,FactValueNumericLow,FactValueNumericHighPrefix,FactValueNumericHigh,Value,FactValueTranslationID,FactComments,Language,DateModified
0,SA_0000001688,"Alcohol, total per capita (15+) consumption (i...",text,SEAR,South-East Asia,Country,BGD,Bangladesh,Year,2022,...,NaN,NaN,0.0009,NaN,0.038,0.0 [0.0 - 0.0],NaN,NaN,EN,2025-05-13T04:00:00.000Z
1,SA_0000001688,"Alcohol, total per capita (15+) consumption (i...",text,EMR,Eastern Mediterranean,Country,SOM,Somalia,Year,2022,...,NaN,NaN,0.0000,NaN,0.000,0.0 [0.0 - 0.0],NaN,NaN,EN,2025-05-13T04:00:00.000Z
2,SA_0000001688,"Alcohol, total per capita (15+) consumption (i...",text,EMR,Eastern Mediterranean,Country,YEM,Yemen,Year,2022,...,NaN,NaN,0.0000,NaN,0.010,0.0 [0.0 - 0.0],NaN,NaN,EN,2025-05-13T04:00:00.000Z
3,SA_0000001688,"Alcohol, total per capita (15+) consumption (i...",text,EMR,Eastern Mediterranean,Country,AFG,Afghanistan,Year,2022,...,NaN,NaN,0.0000,NaN,0.083,0.0 [0.0 - 0.1],NaN,NaN,EN,2025-05-13T04:00:00.000Z
4,SA_0000001688,"Alcohol, total per capita (15+) consumption (i...",text,EMR,Eastern Mediterranean,Country,LBY,Libya,Year,2022,...,NaN,NaN,0.0000,NaN,0.150,0.0 [0.0 - 0.1],NaN,NaN,EN,2025-05-13T04:00:00.000Z


In [4]:
df_alcohol.info()

<class 'pandas.DataFrame'>
RangeIndex: 4324 entries, 0 to 4323
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   IndicatorCode               4324 non-null   str    
 1   Indicator                   4324 non-null   str    
 2   ValueType                   4324 non-null   str    
 3   ParentLocationCode          4324 non-null   str    
 4   ParentLocation              4324 non-null   str    
 5   Location type               4324 non-null   str    
 6   SpatialDimValueCode         4324 non-null   str    
 7   Location                    4324 non-null   str    
 8   Period type                 4324 non-null   str    
 9   Period                      4324 non-null   int64  
 10  IsLatestYear                4324 non-null   bool   
 11  Dim1 type                   4324 non-null   str    
 12  Dim1                        4324 non-null   str    
 13  Dim1ValueCode               4324 non-null   

In [5]:
df_alcohol = df_alcohol.copy()

# 1️⃣ Solo países
df_alcohol = df_alcohol[df_alcohol["Location type"] == "Country"].copy()

In [6]:
df_alcohol = df_alcohol[[
    "Location",
    "SpatialDimValueCode",
    "Period",
    "FactValueNumeric"
]].copy()

In [7]:
df_alcohol = df_alcohol.rename(columns={
    "Location": "country",
    "SpatialDimValueCode": "country_code",
    "Period": "year",
    "FactValueNumeric": "alcohol_consumption"
})

In [8]:
df_alcohol["year"] = df_alcohol["year"].astype(int)
df_alcohol["alcohol_consumption"] = pd.to_numeric(
    df_alcohol["alcohol_consumption"], errors="coerce"
)

In [9]:
df_alcohol = df_alcohol[df_alcohol["year"] >= 2000].copy()

In [10]:
print("Shape final:", df_alcohol.shape)
print("Países únicos:", df_alcohol["country"].nunique())
df_alcohol.head()

Shape final: (4324, 4)
Países únicos: 188


,country,country_code,year,alcohol_consumption
0,Bangladesh,BGD,2022,0.0030
1,Somalia,SOM,2022,0.0000
2,Yemen,YEM,2022,0.0001
3,Afghanistan,AFG,2022,0.0080
4,Libya,LBY,2022,0.0220


In [11]:
# Porcentaje de missing por país
missing_percent = (
    df_alcohol
    .groupby("country")["alcohol_consumption"]
    .apply(lambda x: x.isna().mean() * 100)
    .reset_index(name="missing_percent")
)

threshold = 30  # 30%

countries_to_keep = missing_percent[
    missing_percent["missing_percent"] <= threshold
]["country"]

df_alcohol = df_alcohol[
    df_alcohol["country"].isin(countries_to_keep)
].copy()

print("Países después del filtro:", df_alcohol["country"].nunique())

Países después del filtro: 188


In [12]:
df_alcohol.isna().sum()

country                0
country_code           0
year                   0
alcohol_consumption    0
dtype: int64